In [0]:
--Organization Structure
  --electric_grid
    --data
      --cleaned_data table (parquet file)
      --volume (stored csv data)

CREATE CATALOG IF NOT EXISTS electric_grid;
USE CATALOG electric_grid;
CREATE SCHEMA IF NOT EXISTS electric_grid.data_schema;
USE SCHEMA data_schema;
CREATE VOLUME IF NOT EXISTS volume_set;


You'll need to upload the cleaned, database ready dataset and population dataset into the volume. After running the above SQL query, go to the Catalog tab near the top left (the symbol is three shapes) and open the electric_grid catalog, then data_schema schema, then volume_set volume. Highlight and click the three circles on the volume's tab and click *Upload to Volume*. Find the dataset in your directory and upload. Use the code below to confirm it worked. 

In [0]:
--Parquet File
CREATE OR REPLACE TABLE cleaned_data AS
SELECT * FROM read_files(
  "/Volumes/electric_grid/data_schema/volume_set/doe_events_db_ready.csv",
  format => 'csv',
  header => true,
  inferSchema => true
);

num_affected_rows,num_inserted_rows


In [0]:
%python

#Parquet files offer efficient querying

# Compare CSV and Delta file sizes
csv_path =  "/Volumes/electric_grid/data_schema/volume_set/doe_events_db_ready.csv"

# Get CSV file size
csv_files = dbutils.fs.ls(csv_path)
csv_bytes = sum(f.size for f in csv_files if f.name.endswith('.csv'))

# Get Delta table size from DESCRIBE DETAIL
delta_size = spark.sql("DESCRIBE DETAIL cleaned_data").select("sizeInBytes").first()[0]

# Display comparison
print(f"CSV file size:    {csv_bytes:>15,} bytes  ({csv_bytes / 1024 / 1024:>8.1f} MB)")
print(f"Delta table size: {delta_size:>15,} bytes  ({delta_size / 1024 / 1024:>8.1f} MB)")
print(f"\nCompression ratio: {csv_bytes / delta_size:.1f}x smaller with Delta")

CSV file size:            762,687 bytes  (     0.7 MB)
Delta table size:         111,408 bytes  (     0.1 MB)

Compression ratio: 6.8x smaller with Delta


In [0]:
SELECT * FROM cleaned_data
LIMIT 5;

event_id,year_sheet,event_start_ts,event_end_ts,outage_duration_hours,customers_affected,demand_loss_mw,nerc_region,event_type,alert_criteria,area_affected_raw,event_year,event_month,has_end_ts,has_customers_affected,_rescued_data
1,2002,1/30/2002 6:00,2/7/2002 12:00,198.0,1881134,500.0,SPP,Ice Storm,null,Oklahoma,2002,1,true,true,null
2,2002,1/29/2002 0:00,null,null,270000,500.0,SPP,Ice Storm,null,Metropolitan Kansas City Area,2002,1,false,true,null
3,2002,1/30/2002 16:00,2/10/2002 21:00,269.0,95000,210.0,SPP,Ice Storm,null,Missouri,2002,1,true,true,null
4,2002,2/27/2002 10:48,2/27/2002 11:35,0.783333333,255000,300.0,WSCC,Interruption of Firm Load,null,California,2002,2,true,true,null
5,2002,3/9/2002 0:00,3/11/2002 12:00,60.0,190000,190.0,ECAR,Severe Weather,null,Lower Peninsula of Michigan,2002,3,true,true,null


In [0]:
--@test:outage_duration_greater_than_100
SELECT * FROM cleaned_data
WHERE outage_duration_hours > 100
LIMIT 5;

event_id,year_sheet,event_start_ts,event_end_ts,outage_duration_hours,customers_affected,demand_loss_mw,nerc_region,event_type,alert_criteria,area_affected_raw,event_year,event_month,has_end_ts,has_customers_affected,_rescued_data
1,2002,1/30/2002 6:00,2/7/2002 12:00,198.0,1881134,500.0,SPP,Ice Storm,null,Oklahoma,2002,1,true,true,null
3,2002,1/30/2002 16:00,2/10/2002 21:00,269.0,95000,210.0,SPP,Ice Storm,null,Missouri,2002,1,true,true,null
14,2002,10/3/2002 3:33,10/12/2002 0:00,212.45,242910,null,SPP,Hurricane Lily,null,Coastal Areas of Southern Louisiana,2002,10,true,true,null
18,2002,12/3/2002 18:30,12/9/2002 22:30,148.0,43000,null,SPP,Ice Storm,null,Arkansas,2002,12,true,true,null
20,2002,12/14/2002 11:00,12/19/2002 16:00,125.0,null,180.0,WSCC,Winter Storm,null,Northern and Central California,2002,12,true,false,null


--